# 01 — Quantum Circuit Dataset Generation
**(Notebook 1 / 5 — run this first)**

This notebook demonstrates how to work with quantum circuits and noise models using the `rlnoise` package.
It generates and **saves the canonical datasets** (`datasets/dataset_1q.npz` and `datasets/dataset_3q.npz`) that are required by every subsequent notebook.

1. **Circuit Generation** — Create single and multi-qubit circuits
2. **Noise Models** — Apply and visualise noise effects on circuits
3. **Dataset Generation API** — Walkthrough (demo only, nothing persisted to disk)
4. **Paper Experiment Datasets** — Generate / save the canonical datasets used by all other notebooks

## 1. Setup and Imports

In [1]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import qibo
qibo.set_backend("numpy")

# RL Noise imports
from rlnoise import (
    DatasetConfig,
    NoiseConfig,
    GateSpecificNoise,
    CircuitDataset,
    DatasetGenerator,
    CircuitGenerator,
    CircuitEncoder,
    QuantumNoiseModel,
)

# Import ExperimentConfig for later use
from rlnoise.config import ExperimentConfig

print("Setup complete!")

[Qibo 0.2.23|INFO|2026-04-21 08:23:03]: Using numpy backend on /CPU:0


Setup complete!


## 2. Quantum Circuit Generation

Before creating datasets, let's understand how to generate quantum circuits. We'll explore:
- **Single-qubit vs multi-qubit circuits**
- **Clifford vs non-Clifford circuits**
- **Circuit encoding for machine learning**

### 2.1 Single-Qubit Random Circuits

In [2]:
# Configure a simple single-qubit circuit with three gates
config_1q = DatasetConfig(
    n_circuits=1,
    qubits=1,
    moments=3,
    primitive_gates=["rx", "rz"],
    clifford=False,  # Arbitrary angles (non-Clifford)
)

# Generate a circuit
circuit_gen_1q = CircuitGenerator(config_1q)
circuit_1q = circuit_gen_1q.generate_random_circuit()

print("="*60)
print("  SINGLE-QUBIT CIRCUIT (Non-Clifford)")
print("="*60)
_ = circuit_1q.draw()

# Show gate parameters
print("\nGate Parameters:")
for i, gate in enumerate(circuit_1q.queue):
    if 'theta' in gate.init_kwargs:
        theta = gate.init_kwargs['theta']
        print(f"  Gate {i} ({gate.__class__.__name__}): θ = {theta:.4f} rad = {theta/np.pi:.4f}π")

print("\n** Non-Clifford**: Gates can have any rotation angle between 0 and 2π")

# Show circuit encoding
encoder_1q = CircuitEncoder(config_1q.primitive_gates)
encoded_1q = encoder_1q.circuit_to_array(circuit_1q)

print("\n" + "="*60)
print("  CIRCUIT ENCODING")
print("="*60)
print(f"Shape: {encoded_1q.shape} (moments × qubits × encoding_dim)")
print(f"\nEncoding for each moment:")
for i, moment in enumerate(encoded_1q):
    print(f"  Moment {i}:\n {moment[0]}")

  SINGLE-QUBIT CIRCUIT (Non-Clifford)
0: ─RZ─RX─RZ─

Gate Parameters:
  Gate 0 (RZ): θ = 3.5571 rad = 1.1323π
  Gate 1 (RX): θ = 2.2810 rad = 0.7261π
  Gate 2 (RZ): θ = 2.4949 rad = 0.7942π

** Non-Clifford**: Gates can have any rotation angle between 0 and 2π

  CIRCUIT ENCODING
Shape: (3, 1, 8) (moments × qubits × encoding_dim)

Encoding for each moment:
  Moment 0:
 [1.         0.         0.         0.56612621 0.         0.
 0.         0.        ]
  Moment 1:
 [0.         1.         0.         0.36302708 0.         0.
 0.         0.        ]
  Moment 2:
 [1.         0.         0.         0.39707504 0.         0.
 0.         0.        ]


### 2.2 Single-Qubit Clifford Circuits

In [3]:
# Generate a Clifford circuit
config_1q_clifford = DatasetConfig(
    n_circuits=1,
    qubits=1,
    moments=3,
    primitive_gates=["rx", "rz"],
    clifford=True,  # Quantized angles (Clifford)
)

circuit_gen_clifford = CircuitGenerator(config_1q_clifford)
circuit_clifford = circuit_gen_clifford.generate_random_circuit()

print("="*60)
print("  SINGLE-QUBIT CIRCUIT (Clifford)")
print("="*60)
_ = circuit_clifford.draw()

# Show gate parameters
print("\nGate Parameters:")
for i, gate in enumerate(circuit_clifford.queue):
    if 'theta' in gate.init_kwargs:
        theta = gate.init_kwargs['theta']
        print(f"  Gate {i} ({gate.__class__.__name__}): θ = {theta:.4f} rad = {theta/np.pi:.4f}π")

print("\n**Clifford**: Angles are quantized to {0, π/2, π, 3π/2}")

# Show circuit encoding
encoder_clifford = CircuitEncoder(config_1q_clifford.primitive_gates)
encoded_clifford = encoder_clifford.circuit_to_array(circuit_clifford)

print("\n" + "="*60)
print("  CIRCUIT ENCODING")
print("="*60)
print(f"Shape: {encoded_clifford.shape} (moments × qubits × encoding_dim)")
print(f"\nEncoding for each moment:")
for i, moment in enumerate(encoded_clifford):
    print(f"  Moment {i}:\n {moment[0]}")

  SINGLE-QUBIT CIRCUIT (Clifford)
0: ─RZ─RZ─RZ─

Gate Parameters:
  Gate 0 (RZ): θ = 1.5708 rad = 0.5000π
  Gate 1 (RZ): θ = 4.7124 rad = 1.5000π
  Gate 2 (RZ): θ = 1.5708 rad = 0.5000π

**Clifford**: Angles are quantized to {0, π/2, π, 3π/2}

  CIRCUIT ENCODING
Shape: (3, 1, 8) (moments × qubits × encoding_dim)

Encoding for each moment:
  Moment 0:
 [1.   0.   0.   0.25 0.   0.   0.   0.  ]
  Moment 1:
 [1.   0.   0.   0.75 0.   0.   0.   0.  ]
  Moment 2:
 [1.   0.   0.   0.25 0.   0.   0.   0.  ]


### 2.3 Multi-Qubit Circuits

In [4]:
# Configure a 3-qubit circuit
config_3q = DatasetConfig(
    n_circuits=1,
    qubits=3,
    moments=3,
    primitive_gates=["rx", "rz", "cz"],  # Include CZ gates
    clifford=True,
)

circuit_gen_3q = CircuitGenerator(config_3q)
circuit_3q = circuit_gen_3q.generate_random_circuit()

print("="*60)
print("  3-QUBIT CIRCUIT with Entangling Gates")
print("="*60)
_ = circuit_3q.draw()

# Show circuit encoding
encoder_3q = CircuitEncoder(config_3q.primitive_gates)
encoded_3q = encoder_3q.circuit_to_array(circuit_3q)

print("\n" + "="*60)
print("  CIRCUIT ENCODING")
print("="*60)
print(f"Shape: {encoded_3q.shape} (moments × qubits × encoding_dim)")
for i, moment in enumerate(encoded_3q):
    print(f"  Moment {i}:\n {moment}")



  3-QUBIT CIRCUIT with Entangling Gates
0: ───────Z─
1: ───────|─
2: ─RZ─RX─o─

  CIRCUIT ENCODING
Shape: (3, 3, 8) (moments × qubits × encoding_dim)
  Moment 0:
 [[0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0. ]
 [1.  0.  0.  0.5 0.  0.  0.  0. ]]
  Moment 1:
 [[0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   1.   0.   0.25 0.   0.   0.   0.  ]]
  Moment 2:
 [[ 0.  0. -1.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.  0.  0.]]


## 3. Noise Models

Now let's explore how to apply noise to quantum circuits. Real quantum computers are affected by various types of noise.

### 3.1 Configure Noise Model

Configure the noise parameters for different error types.

In [5]:
# Example of noise configuration using GateSpecificNoise
# The new API allows you to specify exactly which noise channel applies to which gate
from rlnoise import GateSpecificNoise

noise_config = NoiseConfig(
    noise_list=[
        # Apply depolarizing noise to RZ gates
        GateSpecificNoise(gate="rz", noise_channel="depolarizing", noise_parameter=0.02),
        # Apply amplitude damping to RX and RZ gates  
        GateSpecificNoise(gate="rx", noise_channel="damping", noise_parameter=0.03),
        GateSpecificNoise(gate="rz", noise_channel="damping", noise_parameter=0.03),
        # Apply angle-dependent coherent X errors to RX gates
        # A list of two parameters can be used for a circuit with two qubits to specify different noise strengths for each qubit
        GateSpecificNoise(gate="rx", noise_channel="coherent_x", noise_parameter=[0.04, 0.01], angle_dependent=True),
        # Apply angle-dependent coherent Z errors to RZ gates
        # Note: angle_dependent=False means the noise strength is constant and does not depend on the rotation angle of the gate
        GateSpecificNoise(gate="rz", noise_channel="coherent_z", noise_parameter=0.02, angle_dependent=False),
    ]
)

print(noise_config)


  NoiseConfig
  Gate-Specific Noise:
    Gate: rx
      - damping: 0.0300
      - coherent_x: [0.0400, 0.0100] (angle-dependent)
    Gate: rz
      - depolarizing: 0.0200
      - damping: 0.0300
      - coherent_z: 0.0200


### 3.2 Apply Noise to Circuit

Let's see what happens when we apply noise to a circuit.

In [6]:
circuit_1q = circuit_gen_1q.generate_random_circuit()

print("="*60)
print("  ORIGINAL CIRCUIT (No Noise)")
print("="*60)
_ = circuit_1q.draw()
print(f"Gates: {len(circuit_1q.queue)}")

# Apply a depolarizing noise only after RX gates
noise_config_simple = NoiseConfig(
    noise_list=[
        GateSpecificNoise(gate="rx", noise_channel="depolarizing", noise_parameter=0.2),
    ]
)
# Apply the noise model
noise_model = QuantumNoiseModel(noise_config_simple, config_1q.qubits)
noisy_circuit = noise_model.apply(circuit_1q)

print("\n" + "="*60)
print("  NOISY CIRCUIT (Depolarizing after RX only)")
print("="*60)
_ = noisy_circuit.draw()
print(f"Added {len(noisy_circuit.queue) - len(circuit_1q.queue)} noise channels")

# Print all gates in noisy circuit with parameters
print("\n" + "="*60)
print("  ALL GATES IN NOISY CIRCUIT")
print("="*60)
for i, gate in enumerate(noisy_circuit.queue):
    gate_name = gate.__class__.__name__
    qubits = gate.qubits
    print(f"Gate {i}: {gate_name} on qubit(s) {qubits}", end="")
    if 'theta' in gate.init_kwargs:
        theta = gate.init_kwargs['theta']
        gate_type = "noise" if gate.trainable else "original"
        print(f" (θ = {theta:.4f} rad) [{gate_type}]")
    elif 'p0' in gate.init_kwargs:
        p0 = gate.init_kwargs['p0']
        print(f" (p0 = {p0:.4f})")
    elif 'lam' in gate.init_kwargs:
        lam = gate.init_kwargs['lam']
        print(f" (λ = {lam:.4f})")
    else:
        print()

  ORIGINAL CIRCUIT (No Noise)
0: ─RZ─RX─RZ─
Gates: 3

  NOISY CIRCUIT (Depolarizing after RX only)
0: ─RZ─RX─D─RZ─
Added 1 noise channels

  ALL GATES IN NOISY CIRCUIT
Gate 0: RZ on qubit(s) (0,) (θ = 2.5799 rad) [original]
Gate 1: RX on qubit(s) (0,) (θ = 1.3747 rad) [original]
Gate 2: DepolarizingChannel on qubit(s) (0,) (λ = 0.2000)
Gate 3: RZ on qubit(s) (0,) (θ = 4.7648 rad) [original]


In [7]:
# Generate a 3-qubit Clifford circuit
circuit_3q_noisy = circuit_gen_3q.generate_random_circuit()

print("="*60)
print("  ORIGINAL 3-QUBIT CIRCUIT (No Noise)")
print("="*60)
_ = circuit_3q_noisy.draw()
print(f"Gates: {len(circuit_3q_noisy.queue)}")

# Configure noise model with:
# - Amplitude damping after CZ gates
# - Angle-dependent coherent Z rotations after both RX and RZ gates
noise_config_3q = NoiseConfig(
    noise_list=[
        GateSpecificNoise(gate="cz", noise_channel="damping", noise_parameter=0.05),
        GateSpecificNoise(gate="rx", noise_channel="coherent_x", noise_parameter=0.03, angle_dependent=True),
        GateSpecificNoise(gate="rx", noise_channel="coherent_z", noise_parameter=[0.01, 0.02, 0.03], angle_dependent=False),
    ]
)

print("\nNoise configuration:")

# Apply the noise model
noise_model_3q = QuantumNoiseModel(noise_config_3q, config_3q.qubits)
noisy_circuit_3q = noise_model_3q.apply(circuit_3q_noisy)

print("\n" + "="*60)
print("  NOISY 3-QUBIT CIRCUIT")
print("="*60)

_ = noisy_circuit_3q.draw()
print(f"Added {len(noisy_circuit_3q.queue) - len(circuit_3q_noisy.queue)} noise channels")


# Print all gates in noisy circuit with parameters
print("\n" + "="*60)
print("  ALL GATES IN NOISY CIRCUIT")
print("="*60)
for i, gate in enumerate(noisy_circuit_3q.queue):
    gate_name = gate.__class__.__name__
    qubits = gate.qubits
    print(f"Gate {i}: {gate_name} on qubit(s) {qubits}", end="")
    if 'theta' in gate.init_kwargs:
        theta = gate.init_kwargs['theta']
        gate_type = "noise" if gate.trainable else "original"
        print(f" (θ = {theta:.4f} rad) [{gate_type}]")
    elif 'p0' in gate.init_kwargs:
        p0 = gate.init_kwargs['p0']
        print(f" (p0 = {p0:.4f})")
    elif 'lam' in gate.init_kwargs:
        lam = gate.init_kwargs['lam']
        print(f" (λ = {lam:.4f})")
    else:
        print()

  ORIGINAL 3-QUBIT CIRCUIT (No Noise)
0: ─RX─RZ─RX─
1: ─RZ─RX────
2: ─RX─RX────
Gates: 7

Noise configuration:

  NOISY 3-QUBIT CIRCUIT
0: ─RX─RX─RZ─RZ─RX─RX─RZ─
1: ─RZ─RX─RX─RZ──────────
2: ─RX─RX─RZ─RX─RX─RZ────
Added 10 noise channels

  ALL GATES IN NOISY CIRCUIT
Gate 0: RZ on qubit(s) (1,) (θ = 4.7124 rad) [original]
Gate 1: RX on qubit(s) (2,) (θ = 1.5708 rad) [original]
Gate 2: RX on qubit(s) (2,) (θ = 0.0471 rad) [noise]
Gate 3: RZ on qubit(s) (2,) (θ = 0.0300 rad) [noise]
Gate 4: RX on qubit(s) (0,) (θ = 1.5708 rad) [original]
Gate 5: RX on qubit(s) (0,) (θ = 0.0471 rad) [noise]
Gate 6: RZ on qubit(s) (0,) (θ = 0.0100 rad) [noise]
Gate 7: RX on qubit(s) (2,) (θ = 3.1416 rad) [original]
Gate 8: RX on qubit(s) (2,) (θ = 0.0942 rad) [noise]
Gate 9: RZ on qubit(s) (2,) (θ = 0.0300 rad) [noise]
Gate 10: RZ on qubit(s) (0,) (θ = 4.7124 rad) [original]
Gate 11: RX on qubit(s) (1,) (θ = 3.1416 rad) [original]
Gate 12: RX on qubit(s) (1,) (θ = 0.0942 rad) [noise]
Gate 13: RZ on qubit(s

### 3.3 Compare Noisy vs Non-Noisy Circuits

Let's see the direct impact of depolarizing noise on both RX and RZ gates by comparing density matrices.

In [8]:
# Create a simple test circuit with RX and RZ gates
from qibo import gates, Circuit
test_circuit = Circuit(1, density_matrix=True)
test_circuit.add(gates.RX(0, theta=np.pi/4))
test_circuit.add(gates.RZ(0, theta=np.pi/3))
test_circuit.add(gates.RX(0, theta=np.pi/6))

# Execute the clean (non-noisy) circuit
clean_result = test_circuit()
clean_dm = clean_result.state()

# Apply depolarizing noise to both RX and RZ gates
noise_comparison_config = NoiseConfig(
    noise_list=[
        GateSpecificNoise(gate="rx", noise_channel="depolarizing", noise_parameter=0.15),
        GateSpecificNoise(gate="rz", noise_channel="depolarizing", noise_parameter=0.15),
    ]
)

noise_model_compare = QuantumNoiseModel(noise_comparison_config, qubits=1)
noisy_test_circuit = noise_model_compare.apply(test_circuit)

# Execute the noisy circuit
noisy_result = noisy_test_circuit()
noisy_dm = noisy_result.state()

# Compute purities
clean_purity = np.abs(np.trace(clean_dm @ clean_dm))
noisy_purity = np.abs(np.trace(noisy_dm @ noisy_dm))

# Compute difference
dm_difference = noisy_dm - clean_dm
max_difference = np.max(np.abs(dm_difference))
frobenius_distance = np.linalg.norm(dm_difference, 'fro')

# Use Qibo's built-in functions for quantum information metrics
from qibo.quantum_info import fidelity, trace_distance

# Compute trace distance and fidelity using Qibo
td = float(trace_distance(clean_dm, noisy_dm))
fid = fidelity(clean_dm, noisy_dm)


print("\n" + "="*60)
print("  COMPARISON RESULTS")
print("="*60)

print("\nPURITY COMPARISON:")
print(f"  Clean circuit purity:  {clean_purity:.6f}")
print(f"  Noisy circuit purity:  {noisy_purity:.6f}")
print(f"  Purity reduction:      {clean_purity - noisy_purity:.6f}")

print("\nDISTANCE METRICS:")
print(f"  Max absolute difference:     {max_difference:.6f}")
print(f"  Frobenius distance:          {frobenius_distance:.6f}")
print(f"  Trace distance:              {td:.6f}")
print(f"  Fidelity:                    {fid:.6f}")

print("\n  Note: Trace distance ∈ [0, 1], where 0 = identical, 1 = maximally different")
print("        Fidelity ∈ [0, 1], where 1 = identical, 0 = orthogonal")

print("\n" + "="*60)
print("  CLEAN DENSITY MATRIX (No Noise)")
print("="*60)
print(clean_dm)

print("\n" + "="*60)
print("  NOISY DENSITY MATRIX (With Depolarizing)")
print("="*60)
print(noisy_dm)


  COMPARISON RESULTS

PURITY COMPARISON:
  Clean circuit purity:  1.000000
  Noisy circuit purity:  0.688575
  Purity reduction:      0.311425

DISTANCE METRICS:
  Max absolute difference:     0.173671
  Frobenius distance:          0.272855
  Trace distance:              0.192937
  Fidelity:                    0.807063

  Note: Trace distance ∈ [0, 1], where 0 = identical, 1 = maximally different
        Fidelity ∈ [0, 1], where 1 = identical, 0 = orthogonal

  CLEAN DENSITY MATRIX (No Noise)
[[0.71779787+1.38777878e-17j 0.30618622+3.29869804e-01j]
 [0.30618622-3.29869804e-01j 0.28220213+1.38777878e-17j]]

  NOISY DENSITY MATRIX (With Depolarizing)
[[0.63375512+6.93889390e-18j 0.18803661+2.02581294e-01j]
 [0.18803661-2.02581294e-01j 0.36624488+6.93889390e-18j]]


C:\Users\simon\AppData\Local\Temp\ipykernel_12980\3078490167.py:40: ComplexWarning: Casting complex values to real discards the imaginary part
  td = float(trace_distance(clean_dm, noisy_dm))


## 4. Dataset Generation

Now we'll generate complete datasets of circuits with noise for machine learning.

### 4.1 Basic Dataset Generation

Configure dataset parameters and generate training circuits.

In [9]:
# Configure dataset parameters
dataset_config = DatasetConfig(
    n_circuits=50,              # Number of circuits to generate
    qubits=1,                   # Single-qubit circuits
    moments=10,                 # Circuit depth (number of gate layers)
    primitive_gates=["rx", "rz"],  # Available gate types
    clifford=True,              # Use Clifford gates (quantized angles)
    mixed=False,                # All circuits from same distribution
)

print(dataset_config)


  DatasetConfig
    - Circuits:       50
    - Qubits:         1
    - Moments:        10
    - Type:           Clifford
    - Gates:          [rx, rz]



In [10]:
# Create dataset generator and generate dataset
noise_config = NoiseConfig(
    noise_list=[
        # Apply depolarizing noise to RZ gates
        GateSpecificNoise(gate="rz", noise_channel="depolarizing", noise_parameter=0.02),
        # Apply amplitude damping to RX and RZ gates  
        GateSpecificNoise(gate="rx", noise_channel="damping", noise_parameter=0.03),
        GateSpecificNoise(gate="rz", noise_channel="damping", noise_parameter=0.03),
        # Apply angle-dependent coherent X errors to RX gates
        GateSpecificNoise(gate="rx", noise_channel="coherent_x", noise_parameter=0.04, angle_dependent=True),
        # Apply angle-dependent coherent Z errors to RZ gates
        # Note: angle_dependent=False means the noise strength is constant and does not depend on the rotation angle of the gate
        GateSpecificNoise(gate="rz", noise_channel="coherent_z", noise_parameter=0.02, angle_dependent=False),
    ]
)

generator = DatasetGenerator(dataset_config, noise_config)

dataset = generator.generate(verbose=True)

print("\nDataset shape:", dataset.circuits.shape)
print("Dataset labels:", dataset.labels.shape)

print("\nDataset summary:")
print(dataset)

Generating 50 circuits...
Applying noise model...
Encoding circuits...
Dataset generated.

Dataset shape: (50, 10, 1, 8)
Dataset labels: (50, 2, 2)

Dataset summary:

  CircuitDataset
    - Circuits:            50
    - Qubits:              1
    - Moments (depth):     10
    - Encoding dimension:  8
    - Circuit shape:       (10, 1, 8)


### 4.2 Save and Load Dataset (API Demo)

Datasets can be saved to disk and loaded back. This cell demonstrates the API — it writes to a temporary demo path and is **not** the canonical dataset used by subsequent notebooks (that is generated in Section 4 below).

In [11]:
import tempfile, os

# Save to a temporary file just to demonstrate the API
with tempfile.TemporaryDirectory() as tmp_dir:
    save_path = os.path.join(tmp_dir, "dataset_1q_demo")
    dataset.save(save_path)
    print(f"Dataset saved to {save_path}.npz (temporary path — demo only)")

    # Load it back
    loaded_dataset = CircuitDataset.load(save_path + ".npz")
    print(f"Dataset loaded: {len(loaded_dataset)} circuits")

    assert len(loaded_dataset.circuits) == len(dataset.circuits)
    assert len(loaded_dataset.labels)   == len(dataset.labels)
    print("Data integrity verified")
    print(loaded_dataset)

# Train / validation split (in-memory, no save)
train_dataset, val_dataset = dataset.split(val_fraction=0.2)
print(f"\nTrain split: {len(train_dataset)} circuits")
print(f"Val   split: {len(val_dataset)} circuits")

Dataset saved to C:\Users\simon\AppData\Local\Temp\tmpdg566os6\dataset_1q_demo.npz (temporary path — demo only)
Dataset loaded: 50 circuits
Data integrity verified

  CircuitDataset
    - Circuits:            50
    - Qubits:              1
    - Moments (depth):     10
    - Encoding dimension:  8
    - Circuit shape:       (10, 1, 8)

Train split: 40 circuits
Val   split: 10 circuits


### 4.3 Multi-Qubit Dataset

Generate datasets with multi-qubit circuits and two-qubit gates (CZ).

In [12]:
# Configure 3-qubit demo dataset (not saved — demo only)
multiqubit_config = DatasetConfig(
    n_circuits=100,
    qubits=3,
    moments=15,
    primitive_gates=["rx", "rz", "cz"],
    clifford=True,
)

multiqubit_noise = NoiseConfig(
    noise_list=[
        GateSpecificNoise(gate="rx", noise_channel="depolarizing", noise_parameter=0.03),
        GateSpecificNoise(gate="rz", noise_channel="depolarizing", noise_parameter=0.03),
        GateSpecificNoise(gate="cz", noise_channel="depolarizing", noise_parameter=0.03),
        GateSpecificNoise(gate="rx", noise_channel="damping", noise_parameter=0.02),
        GateSpecificNoise(gate="rz", noise_channel="damping", noise_parameter=0.02),
        GateSpecificNoise(gate="rx", noise_channel="coherent_x", noise_parameter=0.04, angle_dependent=True),
        GateSpecificNoise(gate="rz", noise_channel="coherent_z", noise_parameter=0.04, angle_dependent=False),
    ]
)

generator_3q_demo = DatasetGenerator(multiqubit_config, multiqubit_noise)
dataset_3q_demo = generator_3q_demo.generate(verbose=True)

print("\nTraining dataset shape:", dataset_3q_demo.circuits.shape)
print("Training labels shape:", dataset_3q_demo.labels.shape)
print("\nNote: this demo dataset is not saved. The canonical 3-qubit dataset is generated in Section 4.")

Generating 100 circuits...
Applying noise model...
Encoding circuits...
Dataset generated.

Training dataset shape: (100, 15, 3, 8)
Training labels shape: (100, 8, 8)

Note: this demo dataset is not saved. The canonical 3-qubit dataset is generated in Section 4.


## 4. Paper Experiment Datasets

These are the exact configurations used in the published experiments.
Running these cells will generate and save the **canonical datasets** used by all subsequent notebooks:

- `datasets/dataset_1q.npz` — 1-qubit, 100 circuits, depth 10
- `datasets/dataset_3q.npz` — 3-qubit, 800 circuits, depth 20 (mixed)

If the files already exist they are loaded from disk instead, so it is safe to re-run.

> **These datasets are required by `02_gym_environment.ipynb`, `03_training.ipynb`, `04_benchmarking.ipynb` and `05_agent_analysis.ipynb`.**

In [13]:
DATASET_DIR = Path("datasets")
DATASET_DIR.mkdir(exist_ok=True)

### 4.1 1-Qubit Dataset

Noise model from `experiments/1qubit/config.json`:
- Depolarizing λ = 0.02 on **RZ** gates
- Amplitude damping p₀ = 0.03 on **RX** gates
- Coherent-X ε = 0.04 on **RX** gates
- Coherent-Z ε = 0.02 on **RZ** gates

Saved to: **`datasets/dataset_1q.npz`**

In [14]:
DATASET_1Q_PATH = DATASET_DIR / "dataset_1q.npz"

noise_config_1q = NoiseConfig(
    noise_list=[
        GateSpecificNoise(gate="rz", noise_channel="depolarizing", noise_parameter=0.02),
        GateSpecificNoise(gate="rx", noise_channel="damping",      noise_parameter=0.03),
        GateSpecificNoise(gate="rx", noise_channel="coherent_x",   noise_parameter=0.04),
        GateSpecificNoise(gate="rz", noise_channel="coherent_z",   noise_parameter=0.02),
    ]
)

dataset_config_1q = DatasetConfig(
    n_circuits=100,
    moments=10,
    qubits=1,
    primitive_gates=["rx", "rz"],
    clifford=True,
)

if DATASET_1Q_PATH.exists():
    print(f"Loading existing 1-qubit dataset from '{DATASET_1Q_PATH}'...")
    dataset_1q = CircuitDataset.load(str(DATASET_1Q_PATH))
else:
    print("Generating 1-qubit dataset...")
    generator_1q = DatasetGenerator(dataset_config_1q, noise_config_1q)
    dataset_1q = generator_1q.generate(verbose=True)
    dataset_1q.save(str(DATASET_1Q_PATH))
    print(f"Saved to '{DATASET_1Q_PATH}'")

print(dataset_1q)
print(noise_config_1q)

Generating 1-qubit dataset...
Generating 100 circuits...
Applying noise model...
Encoding circuits...
Dataset generated.
Saved to 'datasets\dataset_1q.npz'

  CircuitDataset
    - Circuits:            100
    - Qubits:              1
    - Moments (depth):     10
    - Encoding dimension:  8
    - Circuit shape:       (10, 1, 8)

  NoiseConfig
  Gate-Specific Noise:
    Gate: rx
      - damping: 0.0300
      - coherent_x: 0.0400
    Gate: rz
      - depolarizing: 0.0200
      - coherent_z: 0.0200


### 4.2 3-Qubit Dataset

Noise model from `experiments/3qubit_high/config.json`:
- Depolarizing λ = 0.02 on **RZ** and **CZ** gates
- Amplitude damping p₀ = 0.03 on **RX** and **CZ** gates
- Coherent-X ε = 0.04 on **RX** gates
- Coherent-Z ε = 0.03 on **RZ** gates

800 mixed circuits (half random, half Clifford), depth 20.

Saved to: **`datasets/dataset_3q.npz`**

In [15]:
DATASET_3Q_PATH = DATASET_DIR / "dataset_3q.npz"

noise_config_3q = NoiseConfig(
    noise_list=[
        GateSpecificNoise(gate="rz", noise_channel="depolarizing", noise_parameter=0.02),
        GateSpecificNoise(gate="cz", noise_channel="depolarizing", noise_parameter=0.02),
        GateSpecificNoise(gate="rx", noise_channel="damping",      noise_parameter=0.03),
        GateSpecificNoise(gate="cz", noise_channel="damping",      noise_parameter=0.03),
        GateSpecificNoise(gate="rx", noise_channel="coherent_x",   noise_parameter=0.04),
        GateSpecificNoise(gate="rz", noise_channel="coherent_z",   noise_parameter=0.03),
    ]
)

dataset_config_3q = DatasetConfig(
    n_circuits=800,
    moments=20,
    qubits=3,
    primitive_gates=["rz", "rx", "cz"],
    clifford=False,
    mixed=True,
)

if DATASET_3Q_PATH.exists():
    print(f"Loading existing 3-qubit dataset from '{DATASET_3Q_PATH}'...")
    dataset_3q = CircuitDataset.load(str(DATASET_3Q_PATH))
else:
    print("Generating 3-qubit dataset (this may take a few minutes)...")
    generator_3q_paper = DatasetGenerator(dataset_config_3q, noise_config_3q)
    dataset_3q = generator_3q_paper.generate(verbose=True)
    dataset_3q.save(str(DATASET_3Q_PATH))
    print(f"Saved to '{DATASET_3Q_PATH}'")

print(dataset_3q)
print(noise_config_3q)

Generating 3-qubit dataset (this may take a few minutes)...
Generating 800 circuits...
Applying noise model...
Encoding circuits...
Dataset generated.
Saved to 'datasets\dataset_3q.npz'

  CircuitDataset
    - Circuits:            800
    - Qubits:              3
    - Moments (depth):     31
    - Encoding dimension:  8
    - Circuit shape:       variable (sample: (31, 3, 8))

  NoiseConfig
  Gate-Specific Noise:
    Gate: cz
      - depolarizing: 0.0200
      - damping: 0.0300
    Gate: rx
      - damping: 0.0300
      - coherent_x: 0.0400
    Gate: rz
      - depolarizing: 0.0200
      - coherent_z: 0.0300


## 5. Advanced: JSON Configuration

The `ExperimentConfig` class lets you load / save configurations from / to JSON. This is how the experiment scripts in `experiments/` work.

In [16]:
### 5.1 JSON Configuration

# JSON-style configuration using the new GateSpecificNoise API
config_dict = {
    "dataset": {
        "n_circuits": 100,
        "qubits": 1,
        "moments": 10,
        "primitive_gates": ["rx", "rz"],
        "clifford": True,
    },
    "noise": {
        "noise_list": [
            {
                "gate": "rx",
                "noise_channel": "depolarizing",
                "noise_parameter": 0.02,
                "angle_dependent": False
            },
            {
                "gate": "rz",
                "noise_channel": "depolarizing",
                "noise_parameter": 0.02,
                "angle_dependent": False
            },
            {
                "gate": "rx",
                "noise_channel": "damping",
                "noise_parameter": 0.03,
                "angle_dependent": False
            },
            {
                "gate": "rz",
                "noise_channel": "damping",
                "noise_parameter": 0.03,
                "angle_dependent": False
            },
            {
                "gate": "rx",
                "noise_channel": "coherent_x",
                "noise_parameter": 0.04,
                "angle_dependent": True
            },
            {
                "gate": "rz",
                "noise_channel": "coherent_z",
                "noise_parameter": 0.02,
                "angle_dependent": False
            }
        ]
    }
}

# Create from dictionary
exp_config = ExperimentConfig.from_json(config_dict)
generator_from_json = DatasetGenerator.from_config(exp_config)

# Generate dataset
dataset_from_json = generator_from_json.generate(verbose=False)
print(f"Dataset from JSON config: {len(dataset_from_json)} circuits")

print(dataset_from_json)

Dataset from JSON config: 100 circuits

  CircuitDataset
    - Circuits:            100
    - Qubits:              1
    - Moments (depth):     10
    - Encoding dimension:  8
    - Circuit shape:       (10, 1, 8)
